# 🤖 Sentiment Analysis with BERT

Fine-tuning of a pre-trained **BERT (bert-base-uncased)** model for binary sentiment classification.

### 📊 Dataset
This project uses the **[IMDB Movie Ratings Sentiment Analysis](https://www.kaggle.com/datasets/yasserh/imdb-movie-ratings-sentiment-analysis)** dataset from Kaggle.

The pipeline covers:
1. Data loading and preprocessing
2. Text tokenization via `BertTokenizer`
3. Train / validation / test split
4. Fine-tuning with `AdamW` optimizer and linear learning-rate schedule
5. Evaluation with accuracy, confusion matrix, and classification report

> **Dataset required:** Ensure you have downloaded the dataset from the link above and saved it as a CSV file. Place it in the project root (or update `DATA_PATH` in the *Configuration* cell below).

---
**References:**
- [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)
- [HuggingFace Transformers](https://github.com/huggingface/transformers)

## 📋 Table of Contents

1. [Installation & Imports](#1-installation--imports)
2. [Configuration](#2-configuration)
3. [Data Loading & Preprocessing](#3-data-loading--preprocessing)
4. [Tokenization & Encoding](#4-tokenization--encoding)
5. [DataLoader Setup](#5-dataloader-setup)
6. [Model Initialization](#6-model-initialization)
7. [Training Loop](#7-training-loop)
8. [Training Diagnostics](#8-training-diagnostics)
9. [Inference on Test Set](#9-inference-on-test-set)
10. [Evaluation & Metrics](#10-evaluation--metrics)

## 1 · Installation & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import random
import datetime
import re
import time

# ── Numerical & data handling ─────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Deep learning (PyTorch) ───────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import (
    TensorDataset,
    DataLoader,
    RandomSampler,
    SequentialSampler,
    random_split,
)
from torch.optim import AdamW

# ── Hugging Face Transformers ─────────────────────────────────────────────────
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# ── NLP utilities ─────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Misc ──────────────────────────────────────────────────────────────────────
from tabulate import tabulate

## 2 · Configuration

Centralise every hyper-parameter and path in one place so the notebook is easy to adapt.

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

# ── Paths ─────────────────────────────────────────────────────────────────────
# Update DATA_PATH to point to your dataset CSV file.
# Expected columns: 'text' (str) and 'label' (int, binary 0/1).
DATA_PATH = "../movie.csv"   # <-- change this to your CSV path
MODEL_SAVE_PATH = "bert_model"   # path where the trained model will be saved

# ── Model ─────────────────────────────────────────────────────────────────────
PRETRAINED_MODEL = "bert-base-uncased"  # HuggingFace model hub identifier
NUM_LABELS = 2                          # binary classification

# ── Training hyper-parameters ─────────────────────────────────────────────────
BATCH_SIZE = 16      # fine-tuning batch size (BERT authors suggest 16 or 32)
EPOCHS = 3           # number of training epochs (BERT authors suggest 2–4)
LEARNING_RATE = 2e-5 # AdamW learning rate
ADAM_EPSILON = 1e-8  # AdamW epsilon (numerical stability)
WARMUP_STEPS = 0     # linear warm-up steps for the learning-rate scheduler

# ── Tokenisation ──────────────────────────────────────────────────────────────
MAX_LEN = 311        # maximum token sequence length (set from EDA below)

In [ ]:
# Select GPU if available, otherwise fall back to CPU.
# When running on Colab / a CUDA-enabled machine the output should be 'cuda:0'.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3 · Data Loading & Preprocessing

### 3.1 · Load dataset

In [ ]:
# Load the dataset from the path defined in the Configuration cell.
# The CSV must contain at minimum two columns: 'text' and 'label'.
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.head()

### 3.2 · Shuffle

In [ ]:
def shuffle(df: pd.DataFrame, n: int = 3, random_states: list = [2, 42, 4]) -> pd.DataFrame:
    """
    Apply multiple sequential random shuffles to the DataFrame to break any
    ordering that might bias training.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe to shuffle.
    n : int
        Number of sequential shuffles to apply.
    random_states : list
        Seed values used for each shuffle (one per shuffle).

    Returns
    -------
    pd.DataFrame
        A shuffled copy of the input dataframe.
    """
    df = df.copy()
    for i in range(n):
        df = df.sample(frac=1, random_state=random_states[i])
    return df


new_df = shuffle(df)
new_df.head()

### 3.3 · Text Cleaning

In [ ]:
# Download NLTK stopwords (only required once).
nltk.download('stopwords', quiet=True)
sw = stopwords.words('english')


def clean_text(text: str) -> str:
    """
    Normalise a raw text string by applying the following transformations:
      1. Lowercase conversion.
      2. Remove characters outside the allowed set (letters, ?.!,).
      3. Strip URLs.
      4. Strip HTML tags.
      5. Remove common punctuation characters.
      6. Remove English stop-words.
      7. Remove Unicode emoji characters.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    str
        Cleaned text string.
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Keep only letters and a minimal set of punctuation
    text = re.sub(r"[^a-zA-Z?.!,\xa0]+", " ", text)

    # 3. Remove URLs
    text = re.sub(r"http\S+", "", text)

    # 4. Remove HTML tags
    html = re.compile(r'<.*?>')
    text = html.sub(r'', text)

    # 5. Remove punctuation
    punctuations = '@#!?+&*[]-%.:/();$=><|{}^' + "'`" + '_'
    for p in punctuations:
        text = text.replace(p, '')

    # 6. Remove stop-words
    text = " ".join([word for word in text.split() if word not in sw])

    # 7. Remove emojis
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"   # emoticons
        u"\U0001F300-\U0001F5FF"   # symbols & pictographs
        u"\U0001F680-\U0001F6FF"   # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"   # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub(r'', text)

    return text


# Apply cleaning to the 'text' column in-place.
df['text'] = df['text'].apply(clean_text)
print("Text cleaning complete.")

### 3.4 · Train / Test Split

In [ ]:
# Split the shuffled dataset: 80% training, 20% held-out test.
# We use positional indexing on the shuffled copy (new_df).
split_idx = int(len(df) * 0.8)

train_df = new_df.iloc[:split_idx]
test_df  = new_df.iloc[split_idx:]

print(f"Train size : {len(train_df)} samples")
print(train_df.groupby('label')['text'].count().rename('count'))
print(f"\nTest  size : {len(test_df)} samples")
print(test_df.groupby('label')['text'].count().rename('count'))

## 4 · Tokenization & Encoding

### 4.1 · Load BERT tokenizer

In [ ]:
# Initialise the BERT WordPiece tokenizer.
# do_lower_case=True matches the 'uncased' variant of the model.
tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL, do_lower_case=True)

### 4.2 · Inspect a sample tokenisation

In [ ]:
sentences = train_df.text.values
labels    = list(train_df.label)

# Display the token / token-ID mapping for the first sentence in the training set.
index = 0
sample_sentence = list(sentences)[index]

print(f" Original : {sample_sentence}\n")

tokens    = tokenizer.tokenize(sample_sentence)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

table = np.array([tokens, token_ids]).T
print(tabulate(table, headers=['Token', 'Token ID'], tablefmt='fancy_grid'))

### 4.3 · Token-length distribution (EDA)

In [ ]:
# Compute the token count for every training sentence (including [CLS] & [SEP]).
token_lengths = [
    len(tokenizer.encode(sent, add_special_tokens=True))
    for sent in sentences
]

print(f"Max token length : {max(token_lengths)}")
print(f"Avg token length : {int(sum(token_lengths) / len(token_lengths))}")
print(f"95th percentile  : {int(np.percentile(token_lengths, 95))}")
print(f"99th percentile  : {int(np.percentile(token_lengths, 99))}")

# Visualise the distribution to inform the choice of MAX_LEN.
plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=40, color='steelblue', edgecolor='white')
plt.axvline(np.percentile(token_lengths, 95), color='orange', linestyle='--', label='95th percentile')
plt.axvline(np.percentile(token_lengths, 99), color='red',    linestyle='--', label='99th percentile')
plt.title('Token-length distribution (training set)')
plt.xlabel('Number of tokens')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Secondary check using tokenizer.encode to confirm the max and average lengths.
# This mirrors the logic used in the encoding loop below.
max_len = 0
lengths = []

for sent in list(sentences):
    # Encode with special tokens ([CLS] at start, [SEP] at end).
    input_ids = tokenizer.encode(sent, add_special_tokens=True)
    lengths.append(len(input_ids))
    max_len = max(max_len, len(input_ids))

avg_len = int(sum(lengths) / len(lengths))

print(f"Max sentence length (tokens) : {max_len}")
print(f"Avg sentence length (tokens) : {avg_len}")

### 4.4 · Encode all training sentences

In [ ]:
# Encode the full training set using the BERT tokenizer.
# For each sentence the tokenizer will:
#   (1) Tokenise the text.
#   (2) Prepend [CLS] and append [SEP] special tokens.
#   (3) Map each token to its vocabulary ID.
#   (4) Pad / truncate to MAX_LEN.
#   (5) Build attention masks (1 = real token, 0 = padding).
input_ids      = []
attention_masks = []

for sent in sentences:
    encoded_dict = tokenizer(
        sent,
        add_special_tokens  = True,
        max_length          = MAX_LEN,
        padding             = 'max_length',
        truncation          = True,
        return_attention_mask = True,
        return_tensors      = 'pt',   # return PyTorch tensors
    )
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])

# Stack individual tensors into a single 2-D tensor.
input_ids       = torch.cat(input_ids,       dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels          = torch.tensor(labels)

print(f"input_ids shape       : {input_ids.shape}")
print(f"attention_masks shape : {attention_masks.shape}")
print(f"labels shape          : {labels.shape}")

In [ ]:
def print_rand_sentence_encoding() -> None:
    """Display the tokens, token IDs and attention mask of a random training sample."""
    index       = random.randint(0, len(sentences) - 1)
    tokens      = tokenizer.tokenize(tokenizer.decode(input_ids[index]))
    token_ids_  = [i.numpy() for i in input_ids[index]]
    attention   = [i.numpy() for i in attention_masks[index]]

    table = np.array([tokens, token_ids_, attention]).T
    print(
        tabulate(
            table,
            headers   = ['Token', 'Token ID', 'Attention Mask'],
            tablefmt  = 'fancy_grid',
        )
    )


print_rand_sentence_encoding()

## 5 · DataLoader Setup

In [ ]:
# Combine encoded inputs, attention masks and labels into a single TensorDataset.
dataset = TensorDataset(input_ids, attention_masks, labels)

# 80 / 20 train-validation split (on the already 80% training portion).
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training samples   : {train_size}")
print(f"Validation samples : {val_size}")

# ── DataLoaders ───────────────────────────────────────────────────────────────
# Training data is sampled randomly to prevent ordering bias.
train_dataloader = DataLoader(
    train_dataset,
    sampler    = RandomSampler(train_dataset),
    batch_size = BATCH_SIZE,
)

# Validation data is iterated sequentially (order does not matter for evaluation).
validation_dataloader = DataLoader(
    val_dataset,
    sampler    = SequentialSampler(val_dataset),
    batch_size = BATCH_SIZE,
)

## 6 · Model Initialization

In [ ]:
# Load the pre-trained BERT model with a single linear classification head on top.
# output_attentions and output_hidden_states are disabled to keep the model compact.
model = BertForSequenceClassification.from_pretrained(
    PRETRAINED_MODEL,
    num_labels          = NUM_LABELS,
    output_attentions   = False,
    output_hidden_states = False,
)

# Move the model to the selected device (GPU or CPU).
model = model.to(device)

# Print the total number of trainable parameters.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

In [ ]:
# AdamW is the standard optimiser for BERT fine-tuning.
# Weight decay is handled internally by the Transformers implementation.
optimizer = AdamW(
    model.parameters(),
    lr  = LEARNING_RATE,
    eps = ADAM_EPSILON,
)

In [ ]:
# Total number of optimisation steps = batches × epochs.
total_steps = len(train_dataloader) * EPOCHS

# Linear learning-rate schedule with optional warm-up.
# The LR decreases linearly from LEARNING_RATE to 0 over total_steps.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = WARMUP_STEPS,
    num_training_steps = total_steps,
)

print(f"Total training steps: {total_steps}")

### Helper functions

In [ ]:
def flat_accuracy(preds: np.ndarray, labels: np.ndarray) -> float:
    """
    Compute flat (micro) accuracy by comparing the argmax of each prediction
    row against the corresponding ground-truth label.

    Parameters
    ----------
    preds  : np.ndarray, shape (batch, num_labels)
        Raw logits produced by the model.
    labels : np.ndarray, shape (batch,)
        Ground-truth class indices.

    Returns
    -------
    float
        Fraction of correctly classified samples.
    """
    pred_flat   = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)


def format_time(elapsed: float) -> str:
    """
    Convert an elapsed time (in seconds) to a human-readable hh:mm:ss string.

    Parameters
    ----------
    elapsed : float
        Elapsed time in seconds.

    Returns
    -------
    str
        Formatted time string.
    """
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

## 7 · Training Loop

Each epoch consists of a **training phase** (forward + backward pass, gradient update)
followed by a **validation phase** (inference only, no gradient computation).

In [ ]:
# Fix all random seeds for full reproducibility.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Container to accumulate per-epoch statistics.
training_stats = []

total_t0 = time.time()  # wall-clock timer for the whole run

for epoch_i in range(0, EPOCHS):

    # ──────────────────────────────────────────────────────────────────────────
    #  Training phase
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f" Epoch {epoch_i + 1} / {EPOCHS}")
    print(f"{'='*60}")
    print("  Training...")

    t0 = time.time()
    total_train_loss = 0.0

    # Switch model to training mode (enables dropout, batch-norm updates, etc.).
    model.train()

    for step, batch in enumerate(train_dataloader):

        # Log progress every 40 steps.
        if step % 40 == 0 and step != 0:
            elapsed = format_time(time.time() - t0)
            print(f"    Step {step:>5} / {len(train_dataloader)}  |  Elapsed: {elapsed}")

        # Unpack the batch and move tensors to the selected device.
        b_input_ids, b_input_mask, b_labels = (
            batch[0].to(device),
            batch[1].to(device),
            batch[2].to(device),
        )

        # Zero out any previously accumulated gradients.
        model.zero_grad()

        # Forward pass: compute predictions and cross-entropy loss.
        output = model(
            b_input_ids,
            token_type_ids = None,
            attention_mask = b_input_mask,
            labels         = b_labels,
        )
        loss   = output.loss
        logits = output.logits

        total_train_loss += loss.item()

        # Backward pass: compute gradients.
        loss.backward()

        # Clip gradients to prevent the exploding-gradient problem.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Update model parameters and learning-rate schedule.
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    training_time  = format_time(time.time() - t0)

    print(f"\n  Average training loss : {avg_train_loss:.4f}")
    print(f"  Training epoch time   : {training_time}")

    # ──────────────────────────────────────────────────────────────────────────
    #  Validation phase
    # ──────────────────────────────────────────────────────────────────────────
    print("\n  Running validation...")

    t0 = time.time()

    # Switch to evaluation mode (disables dropout, etc.).
    model.eval()

    total_eval_accuracy = 0.0
    total_eval_loss     = 0.0

    for batch in validation_dataloader:
        b_input_ids, b_input_mask, b_labels = (
            batch[0].to(device),
            batch[1].to(device),
            batch[2].to(device),
        )

        # Disable gradient computation for inference.
        with torch.no_grad():
            output = model(
                b_input_ids,
                token_type_ids = None,
                attention_mask = b_input_mask,
                labels         = b_labels,
            )

        total_eval_loss     += output.loss.item()
        logits               = output.logits.detach().cpu().numpy()
        label_ids            = b_labels.to('cpu').numpy()
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_loss     = total_eval_loss     / len(validation_dataloader)
    validation_time  = format_time(time.time() - t0)

    print(f"  Validation accuracy   : {avg_val_accuracy:.4f}")
    print(f"  Validation loss       : {avg_val_loss:.4f}")
    print(f"  Validation time       : {validation_time}")

    # Store epoch statistics for post-training analysis.
    training_stats.append({
        'epoch'        : epoch_i + 1,
        'Training Loss': avg_train_loss,
        'Valid. Loss'  : avg_val_loss,
        'Valid. Accur.': avg_val_accuracy,
        'Training Time': training_time,
        'Validation Time': validation_time,
    })

print(f"\nTotal training time: {format_time(time.time() - total_t0)}")

# Save the trained model weights to disk.
torch.save(model, MODEL_SAVE_PATH)
print(f"Model saved to '{MODEL_SAVE_PATH}'.")

## 8 · Training Diagnostics

In [ ]:
# Summarise per-epoch statistics in a tidy DataFrame.
pd.set_option('display.precision', 4)

df_stats = pd.DataFrame(data=training_stats)
df_stats = df_stats.set_index('epoch')

# Display the full stats table.
df_stats

In [ ]:
# Plot training vs validation loss curves to diagnose over/under-fitting.
%matplotlib inline

sns.set(style='darkgrid')
sns.set(font_scale=1.5)
plt.rcParams['figure.figsize'] = (12, 6)

plt.plot(df_stats['Training Loss'], 'b-o', label='Training Loss')
plt.plot(df_stats['Valid. Loss'],   'g-o', label='Validation Loss')

plt.title('Training & Validation Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.xticks(df_stats.index)
plt.tight_layout()
plt.show()

## 9 · Inference on Test Set

Load the saved model (or use the one already in memory) and run predictions
over the held-out test split.

In [ ]:
# Load the previously saved model from disk.
# Set weights_only=False to allow loading of the full model object.
model = torch.load(MODEL_SAVE_PATH, weights_only=False)
model = model.to(device)
print(f"Model loaded from '{MODEL_SAVE_PATH}'.")

In [ ]:
# Prepare the test set with the same tokenisation pipeline used for training.
sentences = test_df.text.values
labels    = test_df.label.values

input_ids       = []
attention_masks = []

for sent in sentences:
    encoded_dict = tokenizer(
        sent,
        add_special_tokens    = True,
        max_length            = MAX_LEN,
        padding               = 'max_length',
        truncation            = True,
        return_attention_mask = True,
        return_tensors        = 'pt',
    )
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])

# Stack into 2-D tensors.
input_ids       = torch.cat(input_ids,       dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels          = torch.tensor(labels)

# Build the prediction DataLoader (sequential sampling — order matters for evaluation).
batch_size         = 32
prediction_data    = TensorDataset(input_ids, attention_masks, labels)
prediction_sampler = SequentialSampler(prediction_data)
prediction_dataloader = DataLoader(
    prediction_data,
    sampler    = prediction_sampler,
    batch_size = batch_size,
)

In [ ]:
print(f"Predicting labels for {len(input_ids):,} test sentences...")

model.eval()

predictions  = []
true_labels  = []

for batch in prediction_dataloader:
    # Move tensors to the selected device.
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    # Forward pass without gradient tracking (faster inference, lower memory).
    with torch.no_grad():
        outputs = model(
            b_input_ids,
            token_type_ids = None,
            attention_mask = b_input_mask,
        )

    logits = outputs[0]

    # Move results back to CPU for numpy compatibility.
    predictions.append(logits.detach().cpu().numpy())
    true_labels.append(b_labels.to('cpu').numpy())

print("Prediction complete.")

## 10 · Evaluation & Metrics

In [ ]:
from functools import reduce

# Flatten the list-of-arrays produced by the batched prediction loop.
scores = reduce(lambda x, y: x + y, [list(map(np.argmax, p)) for p in predictions])
labels = reduce(lambda x, y: list(x) + list(y), true_labels)

acc = accuracy_score(scores, labels)
print(f"Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)")

In [ ]:
import itertools


def plot_confusion_matrix(
    cm,
    target_names: list,
    title: str  = 'Confusion Matrix',
    cmap        = None,
    normalize   : bool = True,
) -> None:
    """
    Render a styled confusion matrix plot.

    Parameters
    ----------
    cm : array-like of shape (n_classes, n_classes)
        Confusion matrix from ``sklearn.metrics.confusion_matrix``.
    target_names : list of str
        Human-readable class labels (e.g. ['negative', 'positive']).
    title : str
        Plot title.
    cmap : matplotlib colormap, optional
        Colour map; defaults to 'Blues'.
    normalize : bool
        If True, display proportions; if False, display absolute counts.
    """
    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        fmt = "{:0.4f}" if normalize else "{:,}"
        plt.text(
            j, i, fmt.format(cm[i, j]),
            horizontalalignment='center',
            color='white' if cm[i, j] > thresh else 'black',
        )

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel(
        f'Predicted label\naccuracy={accuracy:.4f}; misclassification={misclass:.4f}'
    )
    plt.show()

In [ ]:
# ── Normalised confusion matrix (proportions) ─────────────────────────────────
sns.set(font_scale=1)
sns.set(style='white')

cm = confusion_matrix(labels, scores)

plot_confusion_matrix(
    cm           = cm,
    normalize    = True,
    target_names = ['Class 0 (Negative)', 'Class 1 (Positive)'],
    title        = 'Confusion Matrix — Normalised',
)

In [ ]:
# ── Detailed classification report ────────────────────────────────────────────
print("=== Classification Report (Precision / Recall / F1-Score) ===")
report = classification_report(
    labels, scores,
    target_names=['Class 0 (Negative)', 'Class 1 (Positive)'],
)
print(report)

# ── Absolute confusion matrix ─────────────────────────────────────────────────
cm_absolute = confusion_matrix(labels, scores)

sns.set(font_scale=1)
sns.set(style='white')

plot_confusion_matrix(
    cm           = cm_absolute,
    normalize    = False,
    target_names = ['Class 0 (Negative)', 'Class 1 (Positive)'],
    title        = 'Confusion Matrix — Absolute Counts',
)